In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# Исходные параметры системы
R_m = 3      # Ом
K_m = 0.02      # В/(рад/с)
J_eq = 2.0e-5    # кг*м^2


print(f"Подбираемые параметры модели:")
print(f"R_m = {R_m:.2f} Ом")
print(f"K_m = {K_m:.3f} В/(рад/с)")
print(f"J_eq = {J_eq:.2e} кг*м^2")

delta = 0.05
print(f"Режим = {delta*100:.0f}%    \n")
K_i_crit = (K_m**3) / (4 * R_m * J_eq)

print(f"Критическое значение Ki_crit = {K_i_crit:.6f}")

# из формулы 29 точное значение K_i
delta_over = 0.1
Ki_max = K_i_crit * (1 + np.pi**2 / (np.log(1.0 / delta_over))**2)
print(f"Ki_max при перерегулировании до {delta_over*100:.0f}%: {Ki_max:.6f}")
print(f"Ki_max / Ki_crit = {(Ki_max / K_i_crit):.6f}")

def analyze_and_plot_responses():
    iteration = 1
    general_time = 4.
    modes = [
        {
            'filename': 'graphic_aperiodic.png',
            't_end': general_time,
            'arr_Ki': [
                ('Ki = 0.85 * Ki_crit', 0.85 * K_i_crit)
            ]
        },
        {
            'filename': 'graphic_critical.png',
            't_end': general_time,
            'arr_Ki': [
                ('Ki = Ki_crit', K_i_crit)
            ]
        },
        {
            'filename': 'graphic_oscillatory.png',
            't_end': general_time,
            'arr_Ki': [
                # ('Ki = 1.5 * Ki_crit', 1.5 * K_i_crit),
                ('Ki = 2.09 * Ki_crit', 2.09 * K_i_crit),
                ('Ki = 2.86 * Ki_crit', 2.86 * K_i_crit),
                #('Ki = 3.0 * Ki_crit', 3.0 * K_i_crit),
                # ('Ki = 4.5 * Ki_crit', 4.5 * K_i_crit),
                ('Ki = 5.0 * Ki_crit', 5.0 * K_i_crit)
            ]
        }
    ]

    for mode in modes:
        t_vec = np.linspace(0, mode['t_end'], 5000)

        plt.figure(figsize=(8, 6))

        for label, Ki in mode['arr_Ki']:
            #формула 17 в отчёте
            num = [Ki * K_m]
            den = [R_m * J_eq, K_m**2, Ki * K_m]
            sys = signal.TransferFunction(num, den)
            #(numerator, denominator) — массивы коэффициентов числителя и знаменателя. Коэффициенты указываются в порядке убывания степеней (например, для s^2 + 3s + 5 передаётся как [1, 3, 5]).

            t, y = signal.step(sys, T=t_vec)

            y_steady = 1.0
            max_val = np.max(y)
            if max_val > y_steady + 1e-7:
                character = "Колебательный"
            else:
                character = "Апериодический"

            # последняя точка вне коридора delta
            out_of_bounds = np.where(np.abs(y - y_steady) > delta)[0]
            t_settling = t[out_of_bounds[-1]] if len(out_of_bounds) > 0 else 0.0

            overshoot = None
            if character == "Колебательный":
                #макс выброс
                idx_max = np.argmax(y)     
                overshoot = (y[idx_max] - y_steady) * 100.0 

                print(f"\n{label}")
                print(f"Характер выхода на константу : {character}")
                print(f"Время выхода в режим ({delta*100:.0f}%) : {t_settling:.4f} с")
                print(f"Максимальный выброс          : {overshoot:.2f}%")

            else:
                print(f"\n{label}")
                print(f"Характер выхода на константу : {character}")
                
            line_label = f"{label}"

            plt.plot(t, y, label=line_label)

            idx_settling = out_of_bounds[-1] if len(out_of_bounds) > 0 else 0
            plt.plot(t[idx_settling], y[idx_settling], 'ko', markersize=4)

            print("Таблица:")
            print(f"K_i = {Ki:.4f} В/(рад*с)  \nВремя выхода в режим: {t_settling:.4f} с  \nУстановившаяся ошибка: {abs(1.0 - y[-1]):.2e}")
            if overshoot is not None:
                print(f"Максимальный выброс: {overshoot:.2f}%\n")


            print("\nCrtl+C в строку таблицы латеха:")
            print(iteration, end="")
            print(f" & {Ki:.4f} & {t_settling:.4f} & ", end="")
            if overshoot is not None:
                print(f"{overshoot:.2f}\\% & ", end="")
            else: print("- & ", end="")
            print(f"{abs(1.0 - y[-1]):.2e}\\\\", end="")
            print("\n\\hline")
            iteration+=1


        plt.axhline(1.0, color='black', linestyle='--', linewidth=1.5, label='Желаемая скорость w*')
        plt.axhspan(1.0 - delta, 1.0 + delta, color='green', alpha=0.25,
                    label=f'{delta*100:.0f}% отклонения от w*')
        plt.xlabel('Время', fontsize=12)
        plt.ylabel('Переходная функция', fontsize=12)
        plt.grid(True, linestyle=':')
        plt.legend(loc='lower right')
        plt.xlim(0, mode['t_end'])
        plt.tight_layout()

        plt.savefig(mode['filename'], dpi=300, bbox_inches='tight')

        plt.show()



In [ ]:
analyze_and_plot_responses()